In [2]:
import pandas as pd

df = pd.read_csv("2019.csv.gz")
df.head(50)

,AE000041196,20190101,TMAX,269,Unnamed: 4,Unnamed: 5,S,Unnamed: 7
0,AE000041196,20190101,TMIN,140,NaN,NaN,S,NaN
1,AE000041196,20190101,TAVG,198,H,NaN,S,NaN
2,AEM00041194,20190101,TMAX,280,NaN,NaN,S,NaN
3,AEM00041194,20190101,TMIN,185,NaN,NaN,S,NaN
4,AEM00041194,20190101,PRCP,0,NaN,NaN,S,NaN


In [12]:
df.head(100000)

,AE000041196,20190101,TMAX,269,Unnamed: 4,Unnamed: 5,S,Unnamed: 7
0,AE000041196,20190101,TMIN,140,NaN,NaN,S,NaN
1,AE000041196,20190101,TAVG,198,H,NaN,S,NaN
2,AEM00041194,20190101,TMAX,280,NaN,NaN,S,NaN
3,AEM00041194,20190101,TMIN,185,NaN,NaN,S,NaN
4,AEM00041194,20190101,PRCP,0,NaN,NaN,S,NaN
...,...,...,...,...,...,...,...,...
99995,ASN00008283,20190102,PRCP,0,NaN,NaN,a,NaN
99996,ASN00008285,20190102,PRCP,0,NaN,NaN,a,NaN
99997,ASN00008286,20190102,PRCP,0,NaN,NaN,a,NaN
99998,ASN00008288,20190102,PRCP,0,NaN,NaN,a,NaN


In [14]:
print(df.columns.tolist())

['AE000041196', '20190101', 'TMAX', '269', 'Unnamed: 4', 'Unnamed: 5', 'S', 'Unnamed: 7']


In [24]:
import pandas as pd
import numpy as np

df_main = pd.read_csv("1819_with_nearest_station.csv")

# 👉 改成你真实的日期列名
DATE_COL = "DATE_COL"

# 统一 station_id / date 类型
df_main["nearest_station_id"] = df_main["nearest_station_id"].astype(str)
df_main["DATE_COL"] = pd.to_datetime(
    df_main["DATE_COL"],
    errors="coerce"
)

print(df_main[[ "nearest_station_id", DATE_COL ]].head())

  nearest_station_id   DATE_COL
0        USC00341743 2018-06-20
1        USC00344258 2018-06-25
2        USC00340917 2018-07-02
3        US1ILSP0027 2018-07-02
4        US1WIMR0003 2018-07-12


In [34]:
def load_and_clean_ghcn(csv_gz_path):
    df = pd.read_csv(csv_gz_path, header=None)

    df.columns = [
        "station_id",
        "date",
        "element",
        "value",
        "mflag",
        "qflag",
        "sflag",
        "obs_time"
    ]

    df["station_id"] = df["station_id"].astype(str)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")

    # ✅ 关键：把 TMAX / TMIN 也加进来
    df = df[df["element"].isin(["PRCP", "TAVG", "TMAX", "TMIN"])]

    # Q-FLAG：只保留为空
    df = df[df["qflag"].isna()]

    # M-FLAG 规则
    allowed = (
        df["mflag"].isna() |
        ((df["mflag"] == "P") & (df["element"] == "PRCP"))
    )
    df = df[allowed]

    # pivot
    df_wide = (
        df
        .pivot_table(
            index=["station_id", "date"],
            columns="element",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    # 单位转换
    for c in ["TAVG", "TMAX", "TMIN"]:
        if c in df_wide.columns:
            df_wide[c] = df_wide[c] / 10.0

    if "PRCP" in df_wide.columns:
        df_wide["PRCP"] = df_wide["PRCP"] / 10.0

    return df_wide

In [36]:
df_2018 = load_and_clean_ghcn("2018.csv.gz")
df_2019 = load_and_clean_ghcn("2019.csv.gz")

df_weather = pd.concat([df_2018, df_2019], ignore_index=True)

# TAVG 补全
df_weather["TAVG_filled"] = df_weather["TAVG"]

mask = (
    df_weather["TAVG"].isna() &
    df_weather["TMAX"].notna() &
    df_weather["TMIN"].notna()
)

df_weather.loc[mask, "TAVG_filled"] = (
    df_weather.loc[mask, "TMAX"] + df_weather.loc[mask, "TMIN"]
) / 2

print(df_weather.head())

element   station_id       date  PRCP  TAVG  TMAX  TMIN  TAVG_filled
0        AE000041196 2018-01-01   NaN   NaN  25.9  11.2        18.55
1        AE000041196 2018-01-06   NaN   NaN  25.4   NaN          NaN
2        AE000041196 2018-01-07   NaN   NaN  27.9   NaN          NaN
3        AE000041196 2018-01-10   NaN   NaN  24.0   NaN          NaN
4        AE000041196 2018-01-11   NaN   NaN  24.2  10.3        17.25


In [39]:
df_weather["date"] = pd.to_datetime(df_weather["date"])
df_weather["year"] = df_weather["date"].dt.year
df_weather["month"] = df_weather["date"].dt.month

In [41]:
def fill_by_month_mean(df, col):
    """
    对某一气象变量：
    在 station_id × (year, month) 维度上
    用该月均值填充缺失值
    """
    return df[col].fillna(
        df.groupby(["station_id", "year", "month"])[col].transform("mean")
    )

In [43]:
for col in ["PRCP", "TMAX", "TMIN", "TAVG"]:
    if col in df_weather.columns:
        df_weather[col] = fill_by_month_mean(df_weather, col)

In [45]:
df_weather["TAVG_filled"] = df_weather["TAVG"]

mask = (
    df_weather["TAVG_filled"].isna() &
    df_weather["TMAX"].notna() &
    df_weather["TMIN"].notna()
)

df_weather.loc[mask, "TAVG_filled"] = (
    df_weather.loc[mask, "TMAX"] + df_weather.loc[mask, "TMIN"]
) / 2

In [47]:
print("缺失率检查：")
print(df_weather[["PRCP", "TMAX", "TMIN", "TAVG", "TAVG_filled"]].isna().mean())

缺失率检查：
element
PRCP           0.081412
TMAX           0.621362
TMIN           0.613845
TAVG           0.904747
TAVG_filled    0.582680
dtype: float64


In [49]:
df_weather

element,station_id,date,PRCP,TAVG,TMAX,TMIN,TAVG_filled,year,month
0,AE000041196,2018-01-01,NaN,NaN,25.9,11.2000,18.55000,2018,1
1,AE000041196,2018-01-06,NaN,NaN,25.4,11.1875,18.29375,2018,1
2,AE000041196,2018-01-07,NaN,NaN,27.9,11.1875,19.54375,2018,1
3,AE000041196,2018-01-10,NaN,NaN,24.0,11.1875,17.59375,2018,1
4,AE000041196,2018-01-11,NaN,NaN,24.2,10.3000,17.25000,2018,1
...,...,...,...,...,...,...,...,...,...
22634903,ZI000067975,2019-08-13,NaN,NaN,NaN,9.5000,NaN,2019,8
22634904,ZI000067975,2019-09-13,NaN,NaN,NaN,12.5000,NaN,2019,9
22634905,ZI000067975,2019-10-09,NaN,NaN,NaN,18.1000,NaN,2019,10
22634906,ZI000067975,2019-11-25,1.0,NaN,NaN,19.2000,NaN,2019,11


In [51]:
core_vars = ["PRCP", "TMAX", "TMIN", "TAVG", "TAVG_filled"]

In [53]:
monthly_missing = (
    df_weather
    .groupby(["station_id", "year", "month"])[core_vars]
    .apply(lambda x: x.isna().all())
    .reset_index()
)

In [55]:
completely_missing_months = monthly_missing[
    monthly_missing[core_vars].all(axis=1)
]

print("完全没有任何气象数据的月份：")
print(completely_missing_months)

完全没有任何气象数据的月份：
Empty DataFrame
Columns: [station_id, year, month, PRCP, TMAX, TMIN, TAVG, TAVG_filled]
Index: []


In [57]:
tavg_missing_months = (
    df_weather
    .groupby(["station_id", "year", "month"])["TAVG_filled"]
    .apply(lambda x: x.isna().all())
    .reset_index(name="TAVG_filled_all_nan")
)

tavg_missing_months = tavg_missing_months[
    tavg_missing_months["TAVG_filled_all_nan"]
]

print(tavg_missing_months)

         station_id  year  month  TAVG_filled_all_nan
21      AE000041196  2019     10                 True
22      AE000041196  2019     11                 True
23      AE000041196  2019     12                 True
45      AEM00041194  2019     10                 True
46      AEM00041194  2019     11                 True
...             ...   ...    ...                  ...
881003  ZI000067975  2019      8                 True
881004  ZI000067975  2019      9                 True
881005  ZI000067975  2019     10                 True
881006  ZI000067975  2019     11                 True
881017  ZI000067983  2018     12                 True

[560188 rows x 4 columns]


In [59]:
import pandas as pd

df_main = pd.read_csv("1819_with_nearest_station.csv")

# 日期统一为 datetime
df_main["DATE_COL"] = pd.to_datetime(
    df_main["DATE_COL"],
    errors="coerce"
)

# station_id 统一为字符串
df_main["nearest_station_id"] = df_main["nearest_station_id"].astype(str)

In [61]:
df_weather["station_id"] = df_weather["station_id"].astype(str)
df_weather["date"] = pd.to_datetime(df_weather["date"])

In [63]:
weather_cols = [
    "station_id",
    "date",
    "PRCP",
    "TAVG",
    "TMAX",
    "TMIN",
    "TAVG_filled"
]

df_weather_sub = df_weather[weather_cols]

In [65]:
df_merged = df_main.merge(
    df_weather_sub,
    left_on=["nearest_station_id", "DATE_COL"],
    right_on=["station_id", "date"],
    how="left"
)

In [67]:
# 删掉重复的 join 键
df_merged = df_merged.drop(columns=["station_id", "date"])

# 把天气变量放到最后（你指定的顺序）
weather_feature_order = ["PRCP", "TAVG", "TMAX", "TMIN", "TAVG_filled"]

other_cols = [c for c in df_merged.columns if c not in weather_feature_order]

df_merged = df_merged[other_cols + weather_feature_order]

In [69]:
df_merged.to_csv(
    "1819_with_nearest_station_weather.csv",
    index=False
)

print("✅ 匹配完成！输出文件：1819_with_nearest_station_weather.csv")
print(df_merged.head())

✅ 匹配完成！输出文件：1819_with_nearest_station_weather.csv
       UID      PHYLUM         CLASS           ORDER           FAMILY  \
0  2014737  ARTHROPODA  MALACOSTRACA        DECAPODA       CAMBARIDAE   
1  2012488  ARTHROPODA       INSECTA         DIPTERA        TABANIDAE   
2  2012489  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
3  2012490  ARTHROPODA       INSECTA         DIPTERA     CHIRONOMIDAE   
4  2012491  ARTHROPODA     ARACHNIDA  TROMBIDIFORMES  TORRENTICOLIDAE   

          GENUS  TARGET_TAXON  TAXA_ID  TOTAL300  IS_DISTINCT300  ...  \
0      FAXONIUS      FAXONIUS     5131       1.0             1.0  ...   
1           NaN     TABANIDAE     4340       1.0             1.0  ...   
2           NaN  CHIRONOMIDAE     3581       2.0             0.0  ...   
3           NaN  CHIRONOMIDAE     3581       NaN             NaN  ...   
4  TORRENTICOLA  TORRENTICOLA     4371       NaN             NaN  ...   

   VISIT_NO  SAMPLE_TYPE MMI_BENT BENT_MMI_COND nearest_station_id  PRCP

In [71]:
stations_in_main = set(df_main["nearest_station_id"].astype(str))
stations_in_weather = set(df_weather["station_id"].astype(str))

missing_stations = stations_in_main - stations_in_weather

print(f"主表中的 station 数: {len(stations_in_main)}")
print(f"天气表中的 station 数: {len(stations_in_weather)}")
print(f"完全不在天气表里的 station 数: {len(missing_stations)}")

list(missing_stations)[:10]

主表中的 station 数: 1748
天气表中的 station 数: 44113
完全不在天气表里的 station 数: 1042


['USW00004967',
 'USC00466293',
 'USC00348720',
 'US1KSOT0014',
 'US1TXWB0037',
 'US1MDBL0030',
 'US1NCNS0034',
 'US1NCFR0002',
 'USC00466250',
 'USC00325610']

In [73]:
stations_in_main = set(df_main["nearest_station_id"].astype(str))
stations_in_weather = set(df_weather["station_id"].astype(str))

missing_stations = stations_in_main - stations_in_weather

print(f"主表中的 station 数: {len(stations_in_main)}")
print(f"天气表中的 station 数: {len(stations_in_weather)}")
print(f"完全不在天气表里的 station 数: {len(missing_stations)}")

list(missing_stations)[:10]

主表中的 station 数: 1748
天气表中的 station 数: 44113
完全不在天气表里的 station 数: 1042


['USW00004967',
 'USC00466293',
 'USC00348720',
 'US1KSOT0014',
 'US1TXWB0037',
 'US1MDBL0030',
 'US1NCNS0034',
 'US1NCFR0002',
 'USC00466250',
 'USC00325610']

In [75]:
no_match_mask = (
    df_merged["PRCP"].isna() &
    df_merged["TAVG_filled"].isna()
)

no_match_df = df_merged[no_match_mask]

print(f"完全没匹配上的记录数: {len(no_match_df)}")
no_match_df[["nearest_station_id", "DATE_COL"]].head(10)

完全没匹配上的记录数: 1549


,nearest_station_id,DATE_COL
1,USC00344258,2018-06-25
2,USC00340917,2018-07-02
3,US1ILSP0027,2018-07-02
4,US1WIMR0003,2018-07-12
5,ACW00011604,2018-07-16
6,US1OKGT0004,2018-07-16
7,US1WIWD0021,2018-07-16
8,US1OKLG0007,2018-07-17
9,ACW00011604,2018-07-17
10,US1WIEC0002,2018-07-17


In [104]:
# df_weather 已包含：
# station_id, date, PRCP, TAVG_filled

weather_index = (
    df_weather
    .set_index(["station_id", "date"])
    .sort_index()
)

In [105]:
df_main = pd.read_csv("1819_with_5_nearest_stations.csv")

df_main["DATE_COL"] = pd.to_datetime(df_main["DATE_COL"])

In [106]:
candidate_cols = [f"candidate_station_{i}_id" for i in range(1, 6)]
df_main["candidate_station_list"] = df_main[candidate_cols].values.tolist()

In [136]:
import numpy as np
import pandas as pd

def get_nearest_date_weather(station_id, target_date):
    """
    在同一 station 内，找日期最近的一天
    """
    try:
        station_df = weather_index.loc[station_id]
    except KeyError:
        return None, None

    if station_df.empty:
        return None, None

    dates = station_df.index.values
    idx = np.argmin(np.abs(dates - np.datetime64(target_date)))
    row = station_df.iloc[idx]

    return row.get("PRCP"), row.get("TAVG_filled")

In [138]:
def resolve_weather_with_fallback(row, max_station_try=5):
    """
    对单条记录做：
    空间（station） × 时间（date） 兜底
    """

    target_date = row["DATE_COL"]
    candidates = row["candidate_station_list"]

    for i, sid in enumerate(candidates[:max_station_try], start=1):

        # ---------- Step 1：同一天 ----------
        try:
            w = weather_index.loc[(sid, target_date)]
            prcp = w.get("PRCP")
            tavg = w.get("TAVG_filled")
        except KeyError:
            prcp, tavg = None, None

        if pd.notna(prcp) or pd.notna(tavg):
            return pd.Series({
                "PRCP_final": prcp,
                "TAVG_final": tavg,
                "weather_station_used": sid,
                "weather_station_rank": i,
                "weather_match_type": "same_day"
            })

        # ---------- Step 2：最近日期 ----------
        prcp, tavg = get_nearest_date_weather(sid, target_date)

        if pd.notna(prcp) or pd.notna(tavg):
            return pd.Series({
                "PRCP_final": prcp,
                "TAVG_final": tavg,
                "weather_station_used": sid,
                "weather_station_rank": i,
                "weather_match_type": "nearest_day"
            })

    # ---------- 完全兜底失败 ----------
    return pd.Series({
        "PRCP_final": None,
        "TAVG_final": None,
        "weather_station_used": None,
        "weather_station_rank": None,
        "weather_match_type": "unresolved"
    })

In [120]:
import numpy as np

df_main["PRCP_final"] = df_main["PRCP"]
df_main["TAVG_final"] = df_main["TAVG_filled"]

df_main["weather_match_type"] = "direct"

# ✅ 用候选列表的第 0 个作为最近站点
df_main["weather_station_used"] = df_main["candidate_station_list"].str[0]
df_main["weather_station_rank"] = 1

In [122]:
df_main

,UID,PHYLUM,CLASS,ORDER,FAMILY,GENUS,TARGET_TAXON,TAXA_ID,TOTAL300,IS_DISTINCT300,...,TAVG,TMAX,TMIN,TAVG_filled,candidate_station_list,PRCP_final,TAVG_final,weather_match_type,weather_station_used,weather_station_rank
0,2014737,ARTHROPODA,MALACOSTRACA,DECAPODA,CAMBARIDAE,FAXONIUS,FAXONIUS,5131,1.0,1.0,...,NaN,28.9,17.2,23.05,"[USC00341743, US1OKRM0003, USC00341738, USC003...",10.2,23.05,direct,USC00341743,1
1,2012488,ARTHROPODA,INSECTA,DIPTERA,TABANIDAE,NaN,TABANIDAE,4340,1.0,1.0,...,NaN,NaN,NaN,NaN,"[USC00344258, US1KSLB0009, USC00140548, USC001...",NaN,NaN,direct,USC00344258,1
2,2012489,ARTHROPODA,INSECTA,DIPTERA,CHIRONOMIDAE,NaN,CHIRONOMIDAE,3581,2.0,0.0,...,NaN,NaN,NaN,NaN,"[USC00340917, USC00418898, USC00349841, USC004...",NaN,NaN,direct,USC00340917,1
3,2012490,ARTHROPODA,INSECTA,DIPTERA,CHIRONOMIDAE,NaN,CHIRONOMIDAE,3581,NaN,NaN,...,NaN,NaN,NaN,NaN,"[US1ILSP0027, USC00471078, US1WIGN0008, US1ILS...",NaN,NaN,direct,US1ILSP0027,1
4,2012491,ARTHROPODA,ARACHNIDA,TROMBIDIFORMES,TORRENTICOLIDAE,TORRENTICOLA,TORRENTICOLA,4371,NaN,NaN,...,NaN,NaN,NaN,NaN,"[US1WIMR0003, US1WIMR0001, USC00478978, USC004...",NaN,NaN,direct,US1WIMR0003,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2101,2014723,ARTHROPODA,INSECTA,EPHEMEROPTERA,NaN,NaN,EPHEMEROPTERA,3708,1.0,0.0,...,NaN,NaN,NaN,NaN,"[US1KSOT0014, US1KSOT0020, US1KSOT0013, US1KSS...",NaN,NaN,direct,US1KSOT0014,1
2102,2014724,MOLLUSCA,BIVALVIA,VENEROIDA,PISIDIIDAE,MUSCULIUM,MUSCULIUM,3997,13.0,1.0,...,NaN,NaN,NaN,NaN,"[US1KSBB0011, US1KSBB0010, USC00148293, US1KSB...",NaN,NaN,direct,US1KSBB0011,1
2103,2014734,ANNELIDA,OLIGOCHAETA,HAPLOTAXIDA,TUBIFICIDAE,AULODRILUS,AULODRILUS,3496,26.0,1.0,...,NaN,NaN,NaN,NaN,"[US1MNSL0105, US1MNSL0076, US1MNSL0263, US1MNS...",NaN,NaN,direct,US1MNSL0105,1
2104,2014736,ANNELIDA,OLIGOCHAETA,HAPLOTAXIDA,NAIDIDAE,NaN,NAIDIDAE,4003,1.0,1.0,...,NaN,NaN,NaN,NaN,"[ACW00011604, US1OKCV0084, US1OKCV0083, US1OKC...",NaN,NaN,direct,ACW00011604,1


In [124]:
mask_need_fallback = (
    df_main["PRCP"].isna() &
    df_main["TAVG_filled"].isna()
)

print("需要兜底的记录数：", mask_need_fallback.sum())

需要兜底的记录数： 1549


In [126]:
def resolve_weather_with_fallback(row, max_station_try=5):

    target_date = row["DATE_COL"]
    candidates = row["candidate_station_list"]

    for i, sid in enumerate(candidates[:max_station_try], start=1):

        # ===== 1. 同日查询 =====
        try:
            w = weather_index.loc[(sid, target_date)]
            prcp = w.get("PRCP")
            tavg = w.get("TAVG_filled")
        except KeyError:
            prcp, tavg = None, None

        if pd.notna(prcp) or pd.notna(tavg):
            return pd.Series({
                "PRCP_final": prcp,
                "TAVG_final": tavg,
                "weather_station_used": sid,
                "weather_station_rank": i,
                "weather_match_type": "same_day"
            })

        # ===== 2. 最近日期 =====
        prcp, tavg = get_nearest_date_weather(sid, target_date)

        if pd.notna(prcp) or pd.notna(tavg):
            return pd.Series({
                "PRCP_final": prcp,
                "TAVG_final": tavg,
                "weather_station_used": sid,
                "weather_station_rank": i,
                "weather_match_type": "nearest_day"
            })

    # ===== 完全失败 =====
    return pd.Series({
        "PRCP_final": None,
        "TAVG_final": None,
        "weather_station_used": None,
        "weather_station_rank": None,
        "weather_match_type": "unresolved"
    })

In [128]:
fallback_result = df_main.loc[mask_need_fallback].apply(
    resolve_weather_with_fallback,
    axis=1
)

for col in fallback_result.columns:
    df_main.loc[mask_need_fallback, col] = fallback_result[col]

In [130]:
df_main["weather_station_rank"].value_counts().sort_index()

weather_station_rank
1.0    745
2.0    709
3.0    267
4.0    160
5.0     96
Name: count, dtype: int64

In [132]:
df_main

,UID,PHYLUM,CLASS,ORDER,FAMILY,GENUS,TARGET_TAXON,TAXA_ID,TOTAL300,IS_DISTINCT300,...,TAVG,TMAX,TMIN,TAVG_filled,candidate_station_list,PRCP_final,TAVG_final,weather_match_type,weather_station_used,weather_station_rank
0,2014737,ARTHROPODA,MALACOSTRACA,DECAPODA,CAMBARIDAE,FAXONIUS,FAXONIUS,5131,1.0,1.0,...,NaN,28.9,17.2,23.05,"[USC00341743, US1OKRM0003, USC00341738, USC003...",10.2,23.05,direct,USC00341743,1.0
1,2012488,ARTHROPODA,INSECTA,DIPTERA,TABANIDAE,NaN,TABANIDAE,4340,1.0,1.0,...,NaN,NaN,NaN,NaN,"[USC00344258, US1KSLB0009, USC00140548, USC001...",2.5,NaN,nearest_day,USC00140548,3.0
2,2012489,ARTHROPODA,INSECTA,DIPTERA,CHIRONOMIDAE,NaN,CHIRONOMIDAE,3581,2.0,0.0,...,NaN,NaN,NaN,NaN,"[USC00340917, USC00418898, USC00349841, USC004...",NaN,NaN,unresolved,None,NaN
3,2012490,ARTHROPODA,INSECTA,DIPTERA,CHIRONOMIDAE,NaN,CHIRONOMIDAE,3581,NaN,NaN,...,NaN,NaN,NaN,NaN,"[US1ILSP0027, USC00471078, US1WIGN0008, US1ILS...",7.6,23.90,same_day,USC00471078,2.0
4,2012491,ARTHROPODA,ARACHNIDA,TROMBIDIFORMES,TORRENTICOLIDAE,TORRENTICOLA,TORRENTICOLA,4371,NaN,NaN,...,NaN,NaN,NaN,NaN,"[US1WIMR0003, US1WIMR0001, USC00478978, USC004...",0.0,NaN,nearest_day,US1WIMR0003,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2101,2014723,ARTHROPODA,INSECTA,EPHEMEROPTERA,NaN,NaN,EPHEMEROPTERA,3708,1.0,0.0,...,NaN,NaN,NaN,NaN,"[US1KSOT0014, US1KSOT0020, US1KSOT0013, US1KSS...",0.0,NaN,same_day,US1KSSA0026,4.0
2102,2014724,MOLLUSCA,BIVALVIA,VENEROIDA,PISIDIIDAE,MUSCULIUM,MUSCULIUM,3997,13.0,1.0,...,NaN,NaN,NaN,NaN,"[US1KSBB0011, US1KSBB0010, USC00148293, US1KSB...",NaN,NaN,unresolved,None,NaN
2103,2014734,ANNELIDA,OLIGOCHAETA,HAPLOTAXIDA,TUBIFICIDAE,AULODRILUS,AULODRILUS,3496,26.0,1.0,...,NaN,NaN,NaN,NaN,"[US1MNSL0105, US1MNSL0076, US1MNSL0263, US1MNS...",19.3,NaN,nearest_day,US1MNSL0105,1.0
2104,2014736,ANNELIDA,OLIGOCHAETA,HAPLOTAXIDA,NAIDIDAE,NaN,NAIDIDAE,4003,1.0,1.0,...,NaN,NaN,NaN,NaN,"[ACW00011604, US1OKCV0084, US1OKCV0083, US1OKC...",0.0,NaN,same_day,US1OKCV0084,2.0
